In [ ]:
from pyspark.sql import SparkSession,functions as f

In [ ]:
spark = SparkSession.builder.appName("Spark_Transformations").master('local[*]').getOrCreate()

In [ ]:
df = spark.read.option("multiLine","true").json('shows__001.json')
df.printSchema()

In [ ]:
def select_show_columns(df):
    return df.select(
        f.col("id").alias("show_id"),
        f.col("name").alias("show_name"),
        f.col("type").alias("show_type"),
        f.col("language").alias("language"),
        f.col("genres").alias("genres"),
        f.col("status").alias("status"),
        f.col("runtime").alias("runtime_minutes"),
        f.col("averageRuntime").alias("average_runtime_minutes"),
        f.col("premiered").alias("premiered_date"),
        f.col("ended").alias("ended_date"),
        f.col("schedule.time").alias("schedule_time"),
        f.col("schedule.days").alias("schedule_days"),
        f.col("rating.average").alias("rating"),
        f.col("network.id").alias("network_id"),
        f.col("network.name").alias("network_name"),
        f.col("network.country.name").alias("network_country"),
        f.col("webchannel.id").alias("webchannel_id"),
        f.col("webchannel.name").alias("webchannel_name"),
        f.col("summary").alias("summary"),
        f.col("updated").alias("updated_at")
    )

selected_df = select_show_columns(df)
print(selected_df.columns)

In [ ]:
def clean_shows(df):
    """
    Cleans textual, categorical, array, and numeric columns in a PySpark DataFrame:
    - Casts and trims text fields.
    - Title-cases categorical fields and fills missing values with 'Unknown'.
    - Replaces null array columns with empty arrays.
    - Strips HTML tags and whitespace from summary text.
    - Converts invalid runtimes (<= 0) to NULL.
    - Deduplicates records based on 'show_id'.
    """

    # List of general string fields that require leading/trailing whitespace removal
    text_columns = [
        "show_name",
        "show_type",
        "language",
        "status",
        "network_name",
        "network_country",
        "webchannel_name"
    ]

    # Categorical string fields to standardize with title casing and default fallbacks
    categorical_columns = [
        "language",
        "network_name",
        "network_country",
        "webchannel_name"
    ]

    # 1. Cast text columns to StringType and trim surrounding whitespace
    for column in text_columns:
        df = df.withColumn(column, f.trim(f.col(column).cast("string")))

    # 2. Capitalize categorical values (Initcap) and replace NULLs with 'Unknown'
    for column in categorical_columns:
        df = df.withColumn(
            column,
            f.initcap(f.coalesce(f.trim(f.col(column)), f.lit("Unknown")))
        )

    # 3. Ensure array columns default to an empty array [] if NULL
    df = df.withColumn(
        'genres',
        f.when(f.col("genres").isNotNull(), f.col('genres')).otherwise(f.array())
    )

    df = df.withColumn(
        'schedule_days',
        f.when(f.col('schedule_days').isNotNull(), f.col("schedule_days")).otherwise(f.array())
    )

    # 4. Remove HTML tags using regex, trim extra whitespace, and handle NULL summaries
    df = df.withColumn(
        "summary",
        f.trim(
            f.regexp_replace(
                f.coalesce(f.col("summary"), f.lit("")),
                r"<[^>]+>",
                ""
            )
        )
    )

    # 5. Replace non-positive runtime values (<= 0) with NULL
    df = df.withColumn(
        'runtime_minutes',
        f.when(f.col('runtime_minutes') <= 0, None).otherwise(f.col("runtime_minutes"))
    )

    df = df.withColumn(
        "average_runtime_minutes",
        f.when(f.col("average_runtime_minutes") <= 0, None).otherwise(f.col("average_runtime_minutes"))
    )

    # 6. Deduplicate records by show_id and report duplicate count
    total_count = df.count()
    df = df.dropDuplicates(["show_id"])
    deduplicated_count = df.count()
    duplicates_count = total_count - deduplicated_count

    if duplicates_count > 0:
        print("Duplicates count: ", duplicates_count)

    return df

cleaned_df = clean_shows(selected_df)

In [ ]:
def convert_datatypes(df):
    """
    Converts column data types and enforces data quality rules in a PySpark DataFrame:
    - Parses string date columns into standard DateType objects.
    - Converts Unix timestamps (in seconds) to TimestampType objects.
    - Casts primary key and foreign key IDs to nullable LongType.
    - Casts runtime and rating columns to DoubleType.
    - Converts out-of-range ratings (< 0 or > 10) to NULL.
    - Converts invalid runtimes (<= 0) to NULL.
    """

    # Converting string dates to Date objects
    df = df.withColumn("premiered_date", f.to_date(f.col("premiered_date")))
    df = df.withColumn("ended_date", f.to_date(f.col("ended_date")))

    # Converting unixtimestamp epoch (in seconds) into TimestampType
    df = df.withColumn("updated_at", f.timestamp_seconds(f.col("updated_at")))

    # Casting ID columns into nullable long datatypes
    df = df.withColumn("network_id", f.col("network_id").cast("long"))
    df = df.withColumn("webchannel_id", f.col("webchannel_id").cast("long"))

    # Casting numerical columns into double datatype
    numerical_columns = ["runtime_minutes", "average_runtime_minutes", "rating"]

    for column in numerical_columns:
        df = df.withColumn(column, f.col(column).cast("double"))

    # Check and set invalid ratings (< 0 or > 10) to NULL
    invalid_ratings_condition = (f.col("rating") < 0) | (f.col("rating") > 10)
    invalid_rating_count = df.filter(invalid_ratings_condition).count()

    if invalid_rating_count > 0:
        df = df.withColumn(
            "rating",
            f.when(invalid_ratings_condition, None).otherwise(f.col("rating"))
        )

    # Check and set invalid runtimes (<= 0) to NULL
    invalid_runtime_condition = f.col("runtime_minutes") <= 0
    invalid_runtime_count = df.filter(invalid_runtime_condition).count()

    if invalid_runtime_count > 0:
        df = df.withColumn(
            "runtime_minutes",
            f.when(invalid_runtime_condition, None).otherwise(f.col("runtime_minutes"))
        )

    return df


converted_df = convert_datatypes(cleaned_df)

In [ ]:
def assign_schedule_ids(df):
    """
    Assigns a stable schedule_id to each show in a PySpark DataFrame based on schedule information:
    - Derives schedule_id directly from show_id if schedule_time or schedule_days exists.
    - Assigns NULL (None) if no schedule information is present.
    - Validates that non-null schedule_id values are strictly unique.
    """

    # 1. Condition: Has schedule_time (non-null and not empty after trimming)
    has_schedule_time = (
        f.col("schedule_time").isNotNull() &
        (f.trim(f.col("schedule_time")) != "")
    )

    # 2. Condition: Has schedule_days (array is not null and has length > 0)
    has_schedule_days = (
        f.col("schedule_days").isNotNull() &
        (f.size(f.col("schedule_days")) > 0)
    )

    # 3. Assign schedule_id = show_id if either condition is true, otherwise NULL
    df = df.withColumn(
        "schedule_id",
        f.when(has_schedule_time | has_schedule_days, f.col("show_id")).otherwise(None)
    )

    # 4. Validation: Verify non-null schedule_ids are unique
    non_null_schedules = df.filter(f.col("schedule_id").isNotNull())

    duplicate_count = (
        non_null_schedules
        .groupBy("schedule_id")
        .count()
        .filter(f.col("count") > 1)
        .count()
    )

    if duplicate_count > 0:
        raise ValueError(f"Duplicate schedule IDs found after assignment: {duplicate_count}")

    unique_schedule_count = non_null_schedules.select("schedule_id").distinct().count()
    print("unique schedule count:", unique_schedule_count)

    return df


scheduled_df = assign_schedule_ids(converted_df)

In [ ]:
def create_derived_fields(df):
    """
    Creates business and analytical derived fields in a PySpark DataFrame:
    - genre_count: Total number of genres in the genres array.
    - has_rating: Boolean flag indicating if a rating exists.
    - show_duration_days: Total days between premiered_date and ended_date.
    - is_currently_running: Boolean flag indicating if status is 'Running'.
    - Replaces negative durations with NULL.
    """

    # 1. Number of genres
    df = df.withColumn(
        "genre_count",
        f.coalesce(f.size(f.col("genres")), f.lit(0))
    )

    # 2. Rating availability (Boolean flag)
    df = df.withColumn("has_rating", f.col("rating").isNotNull())

    # 3. Calculate show duration in days
    df = df.withColumn(
        "show_duration_days",
        f.datediff(f.col("ended_date"), f.col("premiered_date"))
    )

    # 4. Handle invalid (negative) durations by setting them to NULL
    invalid_duration_condition = f.col("show_duration_days") < 0
    invalid_duration_count = df.filter(invalid_duration_condition).count()

    if invalid_duration_count > 0:
        df = df.withColumn(
            "show_duration_days",
            f.when(invalid_duration_condition, None).otherwise(f.col("show_duration_days"))
        )

    # 5. Check if show is currently running (Boolean flag)
    df = df.withColumn("is_currently_running", f.col("status") == f.lit("Running"))
    return df
derived_df = create_derived_fields(scheduled_df)

In [ ]:
derived_df.show(5)

In [ ]:
# Convert PySpark array columns to string first
df_to_save = derived_df \
    .withColumn("genres", f.concat_ws(",", f.col("genres"))) \
    .withColumn("schedule_days", f.concat_ws(",", f.col("schedule_days")))

# Save directly as a single .csv file using Pandas
df_to_save.toPandas().to_csv("processed_shows.csv", index=False)